# Gross Profit and Optimal Discount

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/feature_engineered_data.csv")

PROMO_1_DAYS = 27   # Dec 5–31 2025
PROMO_2_DAYS = 11   # Feb 7–17 2026

# Not including 5 Gallon and Half Pint because despite being included in Promo 2 they were not sold
paint_sizes_p2 = ["Gallon", "Quart"]

# Average Standard Cost per Size 
# Promo 1 — Gallons only
avg_cost_p1 = (
    df[df["Size"] == "Gallon"]
    .groupby("Size")["Standard Cost"]
    .mean()
)

# Promo 2 — All paint sizes
avg_cost_p2 = (
    df[df["Size"].isin(paint_sizes_p2)]
    .groupby("Size")["Standard Cost"]
    .mean()
)

# Average Retail Price per Size (non-promo baseline price) 
avg_retail_p1 = (
    df[df["Size"] == "Gallon"]
    .groupby("Size")["Retail Price"]
    .mean()
)

avg_retail_p2 = (
    df[df["Size"].isin(paint_sizes_p2)]
    .groupby("Size")["Retail Price"]
    .mean()
)

# Average Promo Price per Size (actual Unit Price during promo) 
# Unit Price reflects the discounted price actually charged during the promo
# Both promos were 20% off, so discount_pct = 0.20

PROMO_1_DISCOUNT_PCT = 0.20
PROMO_2_DISCOUNT_PCT = 0.20

avg_promo_price_p1 = avg_retail_p1 * (1 - PROMO_1_DISCOUNT_PCT)
avg_promo_price_p2 = avg_retail_p2 * (1 - PROMO_2_DISCOUNT_PCT)

# Margin per Size 
# Normal margin = avg retail price − avg standard cost (what a day without promo earns per unit)
# Promo margin  = avg promo price  − avg standard cost (what a promo day earns per unit)

margin_p1 = pd.DataFrame({
    "avg_standard_cost":  avg_cost_p1,
    "avg_retail_price":   avg_retail_p1,
    "avg_promo_price":    avg_promo_price_p1,
}).assign(
    normal_margin = lambda x: x["avg_retail_price"] - x["avg_standard_cost"],
    promo_margin  = lambda x: x["avg_promo_price"]  - x["avg_standard_cost"],
)

margin_p2 = pd.DataFrame({
    "avg_standard_cost":  avg_cost_p2,
    "avg_retail_price":   avg_retail_p2,
    "avg_promo_price":    avg_promo_price_p2,
}).assign(
    normal_margin = lambda x: x["avg_retail_price"] - x["avg_standard_cost"],
    promo_margin  = lambda x: x["avg_promo_price"]  - x["avg_standard_cost"],
)

print("=== Promo 1 Margins ===")
print(margin_p1[["avg_standard_cost", "avg_retail_price", "avg_promo_price",
                  "normal_margin", "promo_margin"]].round(2))

print("\n=== Promo 2 Margins ===")
print(margin_p2[["avg_standard_cost", "avg_retail_price", "avg_promo_price",
                  "normal_margin", "promo_margin"]].round(2))

=== Promo 1 Margins ===
        avg_standard_cost  avg_retail_price  avg_promo_price  normal_margin  \
Size                                                                          
Gallon              47.06             92.84            74.27          45.78   

        promo_margin  
Size                  
Gallon         27.22  

=== Promo 2 Margins ===
        avg_standard_cost  avg_retail_price  avg_promo_price  normal_margin  \
Size                                                                          
Gallon              47.06             92.84            74.27          45.78   
Quart               18.80             40.39            32.31          21.59   

        promo_margin  
Size                  
Gallon         27.22  
Quart          13.51  


The first step is to establish the unit economics for each size in each promo. The average standard cost is the cost of goods per unit. The average retail price is the full undiscounted list price for regular customers. The promo price is simply retail multiplied by 0.80 since both promotions were a flat 20% off. Normal margin is what the store earns per unit on a non-promo day, and promo margin is what it earns per unit when the discount is active.

For gallons, the normal margin is $45.78 per unit and the promo margin drops to $27.22, meaning every gallon sold during the promo earns about 40% less per unit than one sold at full price. That is the core trade-off the rest of this notebook works through: can the volume increase from the promotion more than compensate for the margin squeeze? For quarts in Promo 2 the same compression applies, normal margin of $21.59 falls to $13.51 per unit at the 20% discount.

5 Gallon and Half Pint are excluded from the Promo 2 analysis here even though they were technically included in the promotion. The ITS model in notebook 2 found no statistically meaningful lift for either size, and at the unit level neither was sold in significant volume through the Shopify channel during the promo window. Including them would introduce noise without adding any useful signal to the profitability calculation.

Note that the average (cost, regular price, and promo price) is being taken for each size since Product Line and Category data is not available (explained in ReadMe). Additionally, this gross profit calculation counts all sales made during the promo window as a discounted purchase, even though that is not true. The reason for this approximation is that the lift calculated by ITS treats all sales in the promo time frame as a sale made using the promotional discount.

In [ ]:
# Load ITS Baselines & Lifts
lifts_p1 = pd.read_csv("../data/promo1_baselines_and_lifts.csv").set_index("Size")
lifts_p2 = pd.read_csv("../data/promo2_baselines_and_lifts.csv").set_index("Size")

# Keep only the sizes used in gross profit analysis
promo1_gp_sizes = ["Gallon"]
promo2_gp_sizes = ["Gallon", "Quart"]

lifts_p1 = lifts_p1.loc[promo1_gp_sizes]
lifts_p2 = lifts_p2.loc[promo2_gp_sizes]

# Daily Gross Profit During Promo Period 
# Without promo:  baseline units × normal margin
# With promo: (baseline + lift) × promo margin
# Incremental GP: promo GP − baseline GP

def build_gp_table(lifts, margins):
    gp = lifts.join(margins[["normal_margin", "promo_margin"]])

    gp["daily_baseline_gp"] = (
        gp["Expected_Daily_Baseline"] * gp["normal_margin"]
    )

    gp["daily_promo_units"] = (
        gp["Expected_Daily_Baseline"] + gp["Daily_Promo_Lift"]
    )

    gp["daily_promo_gp"] = (
        gp["daily_promo_units"] * gp["promo_margin"]
    )

    gp["daily_incremental_gp"] = (
        gp["daily_promo_gp"] - gp["daily_baseline_gp"]
    )

    return gp

gp_p1 = build_gp_table(lifts_p1, margin_p1)
gp_p2 = build_gp_table(lifts_p2, margin_p2)

print("=== Daily GP During Promo Period — Promo 1 ===")
print(
    gp_p1[
        [
            "Expected_Daily_Baseline",
            "Daily_Promo_Lift",
            "daily_promo_units",
            "daily_baseline_gp",
            "daily_promo_gp",
            "daily_incremental_gp",
        ]
    ].round(2)
)

print("\n=== Daily GP During Promo Period — Promo 2 ===")
print(
    gp_p2[
        [
            "Expected_Daily_Baseline",
            "Daily_Promo_Lift",
            "daily_promo_units",
            "daily_baseline_gp",
            "daily_promo_gp",
            "daily_incremental_gp",
        ]
    ].round(2)
)

=== Daily GP During Promo Period — Promo 1 ===
        Expected_Daily_Baseline  Daily_Promo_Lift  daily_promo_units  \
Size                                                                   
Gallon                     3.51              3.01               6.52   

        daily_baseline_gp  daily_promo_gp  daily_incremental_gp  
Size                                                             
Gallon             160.63          177.41                 16.78  

=== Daily GP During Promo Period — Promo 2 ===
        Expected_Daily_Baseline  Daily_Promo_Lift  daily_promo_units  \
Size                                                                   
Gallon                     4.83              5.45              10.27   
Quart                      2.40              0.96               3.36   

        daily_baseline_gp  daily_promo_gp  daily_incremental_gp  
Size                                                             
Gallon             220.93          279.58                 58.65  
Qua

This cell brings in the ITS baselines and lifts from notebook 2 and computes what daily gross profit looks like with and without the promotion running.

For Promo 1, the baseline of 3.51 gallon units per day at the normal margin of $45.78 produces $160.63 in gross profit per day with no promo active. When the promotion runs, daily units rise to 6.52 (3.51 baseline plus the 3.01 lift), but each unit now only earns the compressed promo margin of $27.22, producing $177.41 in daily promo GP. The incremental daily gain is $16.78, modest, and entirely dependent on that lift estimate being real given its p-value of 0.197 from notebook 2.

For Promo 2 the gallon picture is stronger. The baseline of 4.83 units per day earns $220.93 in daily GP, and the promo pushes volume to 10.27 units generating $279.58, an incremental gain of $58.65 per day. The quart result is the most telling number in this output: the baseline of 2.40 quarts per day earns $51.89, but after the promo the 3.36 units at the compressed margin of $13.51 only generates $45.46, a daily loss of $6.43 relative to doing nothing. This confirms that the Quart discount hurt more than it helped. The lift was not large enough to overcome the margin compression, which is consistent with the Quart lift being statistically insignificant in the ITS model. Running the Quart discount was essentially the store leaving money on the table each day of Promo 2.

In [ ]:
# Total Gross Profit Over Full Promo Period 
gp_p1["total_baseline_gp"] = gp_p1["daily_baseline_gp"] * PROMO_1_DAYS
gp_p1["total_promo_gp"] = gp_p1["daily_promo_gp"] * PROMO_1_DAYS
gp_p1["total_incremental_gp"] = gp_p1["daily_incremental_gp"] * PROMO_1_DAYS

gp_p2["total_baseline_gp"] = gp_p2["daily_baseline_gp"] * PROMO_2_DAYS
gp_p2["total_promo_gp"] = gp_p2["daily_promo_gp"] * PROMO_2_DAYS
gp_p2["total_incremental_gp"] = gp_p2["daily_incremental_gp"] * PROMO_2_DAYS

print(f"=== Total GP Over Promo 1 Period ({PROMO_1_DAYS} days) ===")
print(
    gp_p1[
        ["total_baseline_gp", "total_promo_gp", "total_incremental_gp"]
    ].round(2)
)

print(f"\nPromo 1 Total: Baseline GP    = ${gp_p1['total_baseline_gp'].sum():,.2f}")
print(f"Promo 1 Total: Promo GP       = ${gp_p1['total_promo_gp'].sum():,.2f}")
print(f"Promo 1 Total: Incremental GP = ${gp_p1['total_incremental_gp'].sum():,.2f}")

print(f"\n=== Total GP Over Promo 2 Period ({PROMO_2_DAYS} days) ===")
print(
    gp_p2[
        ["total_baseline_gp", "total_promo_gp", "total_incremental_gp"]
    ].round(2)
)

print(f"\nPromo 2 Total: Baseline GP    = ${gp_p2['total_baseline_gp'].sum():,.2f}")
print(f"Promo 2 Total: Promo GP       = ${gp_p2['total_promo_gp'].sum():,.2f}")
print(f"Promo 2 Total: Incremental GP = ${gp_p2['total_incremental_gp'].sum():,.2f}")

=== Total GP Over Promo 1 Period (27 days) ===
        total_baseline_gp  total_promo_gp  total_incremental_gp
Size                                                           
Gallon            4336.96         4790.01                453.05

Promo 1 Total: Baseline GP    = $4,336.96
Promo 1 Total: Promo GP       = $4,790.01
Promo 1 Total: Incremental GP = $453.05

=== Total GP Over Promo 2 Period (11 days) ===
        total_baseline_gp  total_promo_gp  total_incremental_gp
Size                                                           
Gallon            2430.22         3075.40                645.18
Quart              570.81          500.03                -70.78

Promo 2 Total: Baseline GP    = $3,001.03
Promo 2 Total: Promo GP       = $3,575.44
Promo 2 Total: Incremental GP = $574.41


Multiplying daily GP figures by the number of promo days gives the full-period picture.

For Promo 1 across 27 days, the store would have earned $4,336.96 in gross profit from gallons without the promotion. With the promotion it earned $4,790.01, an incremental gain of $453.05. That is a positive result, but the margin over the no-promo baseline is only about 10%, and it came from giving a 20% discount for an entire month. The low incremental return reflects the two-sided squeeze: the per-unit margin drops nearly 40% while the volume lift, though meaningful, is not large enough to fully compensate at scale (if the promo ran for much longer).

For Promo 2 across 11 days, the combined Gallon and Quart story is more nuanced. Gallons contributed $645.18 in incremental GP which is a strong result for an 11-day window. Quarts offset $70.78 of that, since discounting Quarts at 20% when demand barely responded simply destroyed margin. The net incremental GP across both sizes is $574.41, which is a better result than Promo 1 even though it ran for less than half as many days. This supports the view that February was a much more favorable demand environment for the promotion, and it highlights that the Quart discount was an unnecessary drag. A more targeted Promo 2 that applied the discount only to Gallons would likely have been more profitable.

In [ ]:
from scipy.optimize import minimize_scalar

# Load ITS CIs → Log-Linear PED at Lower / Point / Upper 
# Column names are already set in the CSV headers — no .columns reassignment needed
ci_p1 = pd.read_csv("../data/promo1_its_ci.csv", index_col=0)
ci_p2 = pd.read_csv("../data/promo2_its_ci.csv", index_col=0)

# promo2_its_ci.csv comes from a size-interaction model, so direct per-size
# CI rows are available (promo_Gallon, promo_Quart, etc.) — no scaling needed.

# Extract scalars
baseline_g_p1 = lifts_p1.loc["Gallon", "Expected_Daily_Baseline"]
lift_g_p1     = lifts_p1.loc["Gallon", "Daily_Promo_Lift"]
retail_g_p1   = margin_p1.loc["Gallon", "avg_retail_price"]
cost_g_p1     = margin_p1.loc["Gallon", "avg_standard_cost"]

baseline_g_p2 = lifts_p2.loc["Gallon", "Expected_Daily_Baseline"]
lift_g_p2     = lifts_p2.loc["Gallon", "Daily_Promo_Lift"]
retail_g_p2   = margin_p2.loc["Gallon", "avg_retail_price"]
cost_g_p2     = margin_p2.loc["Gallon", "avg_standard_cost"]

baseline_q_p2 = lifts_p2.loc["Quart", "Expected_Daily_Baseline"]
lift_q_p2     = lifts_p2.loc["Quart", "Daily_Promo_Lift"]
retail_q_p2   = margin_p2.loc["Quart", "avg_retail_price"]  
cost_q_p2     = margin_p2.loc["Quart", "avg_standard_cost"]

# Log-linear PED
# Q(d) = baseline * (1-d)^PED, calibrated so Q(0) = baseline, Q(d_obs) = baseline + lift
# PED = ln(1 + lift/baseline) / ln(1 - d_obs)
# If lift <= 0, treat as inelastic (PED = 0) — no meaningful upward demand response.

def log_linear_ped(lift, baseline, d_obs):
    if lift <= 0 or baseline <= 0:
        return 0.0
    return np.log(1 + lift / baseline) / np.log(1 - d_obs)

# Promo 1: Gallon-only ITS model has a plain is_promo row
# Extracts lower bound, point estimate, and upper bound of sales lift
p1_lift_low  = ci_p1.loc["is_promo", "lower"]
p1_lift_mid  = lift_g_p1
p1_lift_high = ci_p1.loc["is_promo", "upper"]

# Calculates three separate PED scenarios (low, mid, high elasticity) for 
# Promo 1 Gallons by passing the extracted lifts, the baseline, and a constant discount percentage variable
ped_g_p1_low  = log_linear_ped(p1_lift_low,  baseline_g_p1, PROMO_1_DISCOUNT_PCT)
ped_g_p1_mid  = log_linear_ped(p1_lift_mid,  baseline_g_p1, PROMO_1_DISCOUNT_PCT)
ped_g_p1_high = log_linear_ped(p1_lift_high, baseline_g_p1, PROMO_1_DISCOUNT_PCT)

# Promo 2: Size-interaction model gives per-size CI rows directly
p2_g_lift_low  = ci_p2.loc["promo_Gallon", "lower"]
p2_g_lift_high = ci_p2.loc["promo_Gallon", "upper"]
p2_q_lift_low  = ci_p2.loc["promo_Quart",  "lower"]
p2_q_lift_high = ci_p2.loc["promo_Quart",  "upper"]

# Calculates three separate PED scenarios (low, mid, high elasticity) for Gallons
ped_g_p2_low  = log_linear_ped(p2_g_lift_low,  baseline_g_p2, PROMO_2_DISCOUNT_PCT)
ped_g_p2_mid  = log_linear_ped(lift_g_p2,       baseline_g_p2, PROMO_2_DISCOUNT_PCT)
ped_g_p2_high = log_linear_ped(p2_g_lift_high,  baseline_g_p2, PROMO_2_DISCOUNT_PCT)

# Calculates three separate PED scenarios (low, mid, high elasticity) for Quarts
ped_q_p2_low  = log_linear_ped(p2_q_lift_low,  baseline_q_p2, PROMO_2_DISCOUNT_PCT)
ped_q_p2_mid  = log_linear_ped(lift_q_p2,       baseline_q_p2, PROMO_2_DISCOUNT_PCT)
ped_q_p2_high = log_linear_ped(p2_q_lift_high,  baseline_q_p2, PROMO_2_DISCOUNT_PCT)

print("=== Log-Linear PED Summary ===")
print(f"\nPromo 1 — Gallon:")
print(f"  Lower CI lift  {p1_lift_low:+.2f}  →  PED = {ped_g_p1_low:.4f}{'  (inelastic: lift ≤ 0)' if p1_lift_low <= 0 else ''}")
print(f"  Point est      {p1_lift_mid:+.2f}  →  PED = {ped_g_p1_mid:.4f}")
print(f"  Upper CI lift  {p1_lift_high:+.2f}  →  PED = {ped_g_p1_high:.4f}")

print(f"\nPromo 2 — Gallon (direct from size-interaction ITS CI):")
print(f"  Lower CI lift  {p2_g_lift_low:+.2f}  →  PED = {ped_g_p2_low:.4f}")
print(f"  Point est      {lift_g_p2:+.2f}  →  PED = {ped_g_p2_mid:.4f}")
print(f"  Upper CI lift  {p2_g_lift_high:+.2f}  →  PED = {ped_g_p2_high:.4f}")

print(f"\nPromo 2 — Quart (direct from size-interaction ITS CI):")
print(f"  Lower CI lift  {p2_q_lift_low:+.2f}  →  PED = {ped_q_p2_low:.4f}{'  (inelastic: lift ≤ 0)' if p2_q_lift_low <= 0 else ''}")
print(f"  Point est      {lift_q_p2:+.2f}  →  PED = {ped_q_p2_mid:.4f}")
print(f"  Upper CI lift  {p2_q_lift_high:+.2f}  →  PED = {ped_q_p2_high:.4f}")

=== Log-Linear PED Summary ===

Promo 1 — Gallon:
  Lower CI lift  -1.61  →  PED = 0.0000  (inelastic: lift ≤ 0)
  Point est      +3.01  →  PED = -2.7762
  Upper CI lift  +7.63  →  PED = -5.1766

Promo 2 — Gallon (direct from size-interaction ITS CI):
  Lower CI lift  +1.45  →  PED = -1.1798
  Point est      +5.45  →  PED = -3.3861
  Upper CI lift  +7.18  →  PED = -4.0856

Promo 2 — Quart (direct from size-interaction ITS CI):
  Lower CI lift  -2.15  →  PED = 0.0000  (inelastic: lift ≤ 0)
  Point est      +0.96  →  PED = -1.5067
  Upper CI lift  +3.58  →  PED = -4.0851


Price elasticity of demand is the input the optimizer needs. It answers the question: if I change the discount percentage, how does demand respond? The log-linear model used here calibrates a PED from the observed data given that a 20% discount produced a lift of X units above a baseline of Y, what elasticity would explain that response? The formula is PED = ln(1 + lift/baseline) / ln(1 - discount_pct). The result is always negative for a normal good, meaning price decreases lead to quantity increases.

Three scenarios are computed for each size using the ITS confidence interval: the lower bound lift gives the least elastic (most conservative) scenario, the point estimate gives the expected scenario, and the upper bound lift gives the most elastic (most optimistic) scenario. This is how the uncertainty from notebook 2 gets carried forward; instead of using a single PED assumption, the optimizer runs across the full plausible range.

The Promo 1 lower CI lift of -1.61 is negative, meaning the pessimistic scenario says the promotion had no demand effect at all. Negative or zero lifts are treated as PED = 0 — perfectly inelastic demand — which means any discount just destroys margin with no volume reward. In that scenario the optimizer will correctly conclude the best discount is zero percent. The point estimate PED of -2.78 means a 1% price decrease is associated with roughly a 2.78% increase in quantity demanded, which is moderately elastic. The upper CI PED of -5.18 represents a highly elastic scenario where demand is very sensitive to the discount.

For Promo 2 gallons, even the lower CI lift of +1.45 is positive, so all three scenarios produce meaningful negative PED values ranging from -1.18 to -4.09. This is why the Promo 2 gallon results in the optimizer will look more consistent than Promo 1 — there is less uncertainty about the direction of the demand response. For Promo 2 quarts, the lower CI lift is negative so it collapses to PED = 0, while the point estimate of -1.51 indicates moderate elasticity. The fact that the quart elasticity range spans from 0 to -4.09 is another reflection of how noisy and unreliable the quart lift estimate was.

In [ ]:
# Optimization with Log-Linear Demand
# Q(d) = baseline * (1-d)^PED
# M(d) = retail * (1-d) - cost
# Daily GP = Q(d) * M(d)

# Function factory that creates a gross profit function for a given PED scenario
# SciPy's optimize_scalar function requires a function that takes a single argument and returns a scalar
def make_gp_fn(baseline, retail, cost, ped):
    # Returns gross profit for discount d
    def gp(d):
        return baseline * (1 - d) ** ped * (retail * (1 - d) - cost)
    return gp

# Optimizes discount d to maximize gross profit
def optimize_discount(gp_fn, bounds=(0, 0.50)):
    res = minimize_scalar(lambda d: -gp_fn(d), bounds=bounds, method="bounded")
    return res.x, gp_fn(res.x)

# Promo 1: 3 CI scenarios
p1_scenarios = {
    "Lower CI (less elastic)": ped_g_p1_low,
    "Point estimate":          ped_g_p1_mid,
    "Upper CI (more elastic)": ped_g_p1_high,
}

print(f"=== Promo 1 Optimization — Gallon ({PROMO_1_DAYS} days) ===")
print(f"  {'Scenario':<28} {'Opt Discount':>13} {'Max Daily GP':>13} {'Total GP':>11}")
print("  " + "-" * 70)

# For each scenario, creates a gross profit function, optimizes the discount, and prints the results
for label, ped in p1_scenarios.items():
    gp_fn = make_gp_fn(baseline_g_p1, retail_g_p1, cost_g_p1, ped)
    opt_d, max_dgp = optimize_discount(gp_fn)
    print(f"  {label:<28} {opt_d:>12.1%} ${max_dgp:>11,.2f} ${max_dgp * PROMO_1_DAYS:>9,.2f}")

# Creates a gross profit function for the midpoint PED scenario (no-promo baseline)
gp_mid_p1  = make_gp_fn(baseline_g_p1, retail_g_p1, cost_g_p1, ped_g_p1_mid)
print(f"\n  Actual 20% daily GP   : ${gp_mid_p1(0.20):,.2f}  (total: ${gp_mid_p1(0.20) * PROMO_1_DAYS:,.2f})")
print(f"  No-promo baseline GP  : ${gp_mid_p1(0.00):,.2f}")

# Promo 2: 3 CI levels × 2 Quart scenarios 
# Quart sensitivity:
#   "Point est. PED" — use the observed Quart lift to estimate elasticity
#   "Inelastic (PED=0)" — treat the Quart lift as noise; Quart demand doesn't
#     respond to the discount, so any discount only compresses margin → optimizer
#     will push the combined discount lower to protect Quart profitability

p2_ci_scenarios = {
    "Lower CI":      (ped_g_p2_low,  ped_q_p2_low),
    "Point estimate":(ped_g_p2_mid,  ped_q_p2_mid),
    "Upper CI":      (ped_g_p2_high, ped_q_p2_high),
}

# Quart sensitivity test: What happens if I treat the Quart lift as noise?
quart_sensitivity = {
    "Quart = point est. PED": "point",
    "Quart = inelastic (PED=0)": "zero",
}

print(f"\n=== Promo 2 Optimization — Gallon + Quart ({PROMO_2_DAYS} days) ===")

# For each Quart sensitivity scenario, iterates through each CI scenario
for q_label, q_mode in quart_sensitivity.items():
    print(f"\n  [ {q_label} ]")
    print(f"  {'CI Scenario':<20} {'Opt Discount':>13} {'Max Daily GP':>13} {'Total GP':>11}")
    print("  " + "-" * 62)

    # For each CI scenario, creates a gross profit function for Gallons and Quarts,
    # and then combines them to create a single gross profit function for the entire promotion.
    for ci_label, (ped_g, ped_q_pt) in p2_ci_scenarios.items():
        ped_q = 0.0 if q_mode == "zero" else ped_q_pt

        # Capture loop vars in default args to avoid closure bug
        # Creates a gross profit function for Gallons and Quarts
        gp_g = make_gp_fn(baseline_g_p2, retail_g_p2, cost_g_p2, ped_g)
        gp_q = make_gp_fn(baseline_q_p2, retail_q_p2, cost_q_p2, ped_q)

        # Creates a combined gross profit function that adds the gross profit functions for Gallons and Quarts
        def combined_gp(d, _g=gp_g, _q=gp_q):
            return _g(d) + _q(d)

        # Optimizes the combined gross profit function to find the optimal discount
        opt_d, max_dgp = optimize_discount(combined_gp)
        print(f"  {ci_label:<20} {opt_d:>12.1%} ${max_dgp:>11,.2f} ${max_dgp * PROMO_2_DAYS:>9,.2f}")

gp_g_mid = make_gp_fn(baseline_g_p2, retail_g_p2, cost_g_p2, ped_g_p2_mid)
gp_q_mid = make_gp_fn(baseline_q_p2, retail_q_p2, cost_q_p2, ped_q_p2_mid)
actual_20_p2  = gp_g_mid(0.20) + gp_q_mid(0.20)
baseline_p2   = gp_g_mid(0.00) + gp_q_mid(0.00)

print(f"\n  Actual 20% daily GP   : ${actual_20_p2:,.2f}  (total: ${actual_20_p2 * PROMO_2_DAYS:,.2f})")
print(f"  No-promo baseline GP  : ${baseline_p2:,.2f}")

=== Promo 1 Optimization — Gallon (27 days) ===
  Scenario                      Opt Discount  Max Daily GP    Total GP
  ----------------------------------------------------------------------
  Lower CI (less elastic)              0.0% $     160.63 $ 4,336.92
  Point estimate                      20.8% $     177.45 $ 4,791.12
  Upper CI (more elastic)             37.2% $     438.56 $11,841.03

  Actual 20% daily GP   : $177.41  (total: $4,790.01)
  No-promo baseline GP  : $160.63

=== Promo 2 Optimization — Gallon + Quart (11 days) ===

  [ Quart = point est. PED ]
  CI Scenario           Opt Discount  Max Daily GP    Total GP
  --------------------------------------------------------------
  Lower CI                     0.0% $     272.82 $ 3,001.02
  Point estimate              26.4% $     331.52 $ 3,646.74
  Upper CI                    33.9% $     477.80 $ 5,255.83

  [ Quart = inelastic (PED=0) ]
  CI Scenario           Opt Discount  Max Daily GP    Total GP
  ----------------------

The optimizer finds the discount percentage that maximizes gross profit given the demand model Q(d) = baseline × (1-d)^PED and the per-unit margin M(d) = retail × (1-d) - cost. As the discount increases, two things happen simultaneously: volume rises because of the elasticity effect, and per-unit margin falls because the price is lower. The optimal discount is the point where those two forces exactly balance.

For Promo 1, the results show how dramatically the optimal discount shifts depending on the elasticity assumption. At PED = 0 (lower CI, inelastic), the optimizer recommends 0% — no discount at all. In this scenario the promotion generated $4,337 total GP versus $4,337 with no promo (they are equal because no discount means no margin loss), so the store would have been equally well off doing nothing. At the point estimate PED of -2.78, the optimal discount is 20.8%, which is essentially the exact 20% that was actually offered. This is a meaningful validation: the actual discount used in Promo 1 was right at the optimum under the most likely demand scenario, even though no formal optimization was done at the time. At the upper CI PED of -5.18, demand is highly elastic and the optimizer suggests 37.2% could have generated $11,841 total, but this requires the true lift to be at the very top of a very wide confidence interval.

For Promo 2, I ran two sensitivity tests on the Quart assumption to see whether it mattered whether Quart demand was truly elastic or just noise. When I treat Quart demand as elastic (point estimate PED), the optimizer recommends a 26.4% discount at the point estimate scenario and calculates $3,647 total GP — a modest improvement over the $3,575 actually achieved with 20%. When I treat Quart demand as completely inelastic (PED = 0), the optimal discount drops slightly to 25.6% because the optimizer no longer sees any volume benefit from discounting Quarts and pushes the combined discount lower to protect the Quart margin. The difference between the two Quart assumptions is relatively small at the point estimate level, which tells me the Gallon demand response is the dominant driver of Promo 2 profitability and the Quart assumption barely moves the needle either way.

The practical conclusion from the optimization is that the 20% discount applied in both promotions was a defensible choice. For Promo 1, 20% lands almost exactly at the theoretical optimum under the most likely demand scenario. For Promo 2, there is a credible case that a slightly higher discount in the 25-27% range could have improved profitability, particularly on gallons where the demand signal is strong and statistically significant. The high-uncertainty upper CI scenarios for both promos suggest aggressive discounts could generate large returns, but those scenarios require lifts at the top of wide confidence intervals and should not be taken as reliable forecasts. The lower CI scenarios serve as a useful stress test: they show that if demand were barely responsive, both promotions would have been essentially break-even or value-destructive, reinforcing the importance of measuring and validating the lift before scaling a promotion.